# 01 — Data Pipeline

Constructs IV² and RV² panels from OptionMetrics and TAQ data.

**Outputs saved to `data/`:**
- `iv2_panel.parquet` — model-free implied variance, monthly, (dates × tickers)
- `rv2_panel.parquet` — realized variance, monthly, (dates × tickers)
- `vrp_panel.parquet` — VRP = IV² − RV²
- `returns_panel.parquet` — monthly stock returns
- `market_cap_panel.parquet` — month-end market capitalizations

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, '..')
from src.vrp import build_vrp_panel

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## 1. Load OptionMetrics data

Requires WRDS access. Substitute your own loader below.

```python
import wrds
db = wrds.Connection()
om = db.raw_sql("""
    SELECT secid, date, strike_price, best_bid, best_offer,
           impl_volatility, cp_flag, expiration, forward_price
    FROM optionm.opprcd
    WHERE date >= '1996-01-01' AND ss_flag = 0
""", date_cols=['date','expiration'])
```

In [15]:
# Synthetic demo data (replace with real OptionMetrics + TAQ)
np.random.seed(42)
n_stocks, n_months = 100, 300
dates = pd.date_range('2000-01-31', periods=n_months, freq='ME')
tickers = [f'TICK{i:03d}' for i in range(n_stocks)]

# IV² ~ 0.04 (approx 20% annualised vol), VRP ~ 0.005
iv2 = pd.DataFrame(
    np.abs(np.random.normal(0.04, 0.012, (n_months, n_stocks))),
    index=dates, columns=tickers
)
rv2 = iv2 - pd.DataFrame(
    np.random.normal(0.005, 0.003, (n_months, n_stocks)),
    index=dates, columns=tickers
)
print(f'IV² panel: {iv2.shape}  mean={iv2.mean().mean():.4f}')
print(f'RV² panel: {rv2.shape}  mean={rv2.mean().mean():.4f}')

IV² panel: (300, 100)  mean=0.0400
RV² panel: (300, 100)  mean=0.0350


## 2. Compute VRP panel

In [16]:
vrp = build_vrp_panel(iv2, rv2)
print(f'VRP mean: {vrp.mean().mean():.4f}  std: {vrp.std().mean():.4f}')
vrp.describe().T.head()

VRP mean: 0.0050  std: 0.0030


,count,mean,std,min,25%,50%,75%,max
TICK000,300.0,0.005136,0.002943,-0.003857,0.003143,0.005216,0.007008,0.012680
TICK001,300.0,0.004951,0.002828,-0.003255,0.002973,0.005076,0.006988,0.012592
TICK002,300.0,0.004856,0.003033,-0.004836,0.002750,0.004889,0.006893,0.013053
TICK003,300.0,0.004898,0.003079,-0.004970,0.002689,0.004857,0.007140,0.012754
TICK004,300.0,0.004828,0.002893,-0.004610,0.002948,0.005053,0.006746,0.011826


## 3. Load returns and market cap

In [17]:
returns = pd.DataFrame(
    np.random.normal(0.008, 0.06, (n_months, n_stocks)),
    index=dates, columns=tickers
)

# Inject decaying VRP signal: strong pre-2009, fading after
for t in range(n_months - 1):
    strength = max(0.0, 0.5 - 0.004 * max(0, t - 108))
    vrp_rank = vrp.iloc[t].rank(pct=True) - 0.5
    returns.iloc[t + 1] += strength * vrp_rank * 0.02

market_cap = pd.DataFrame(
    np.exp(np.random.normal(8, 1.5, (n_months, n_stocks))),
    index=dates, columns=tickers
)

## 4. Save panels

In [18]:
iv2.to_parquet(DATA_DIR / 'iv2_panel.parquet')
rv2.to_parquet(DATA_DIR / 'rv2_panel.parquet')
vrp.to_parquet(DATA_DIR / 'vrp_panel.parquet')
returns.to_parquet(DATA_DIR / 'returns_panel.parquet')
market_cap.to_parquet(DATA_DIR / 'market_cap_panel.parquet')
print('All panels saved.')

All panels saved.
